# [7.5] Predictive Concept Decoders - Solutions

Reference validation notebook for the section-local mini PCD implementation.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter7_activation_to_language"
section = "part5_predictive_concept_decoders"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_predictive_concept_decoders.tests as tests
from chapter7_activation_to_language.exercises.part5_predictive_concept_decoders import solutions

In [ ]:
tests.test_build_pcd_question_batch_validates_shapes_and_questions(
    solutions.build_pcd_question_batch,
    solutions.default_pcd_questions,
)
tests.test_sparse_concept_encode_and_sparsity_controls(
    solutions.sparse_concept_encode,
    solutions.concept_sparsity_report,
)
tests.test_question_conditioned_decoder_uses_question_information(
    solutions.question_conditioned_decoder_logits,
)
tests.test_question_conditioned_decoder_respects_arbitrary_weight_and_bias(
    solutions.question_conditioned_decoder_logits,
)
tests.test_trained_question_conditioned_decoder_learns_concept_question_interaction(
    solutions.question_conditioned_concept_features,
    solutions.train_question_conditioned_decoder,
    solutions.question_conditioned_decoder_logits,
)
tests.test_pcd_comparison_report_beats_baselines(
    solutions.pcd_comparison_report,
)
tests.test_pcd_comparison_report_scores_each_baseline_independently(
    solutions.pcd_comparison_report,
)
tests.test_concept_stability_removal_and_audit_controls(
    solutions.concept_stability_report,
    solutions.concept_removal_report,
    solutions.concept_audit_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["batch"]["num_questions"] == 4
assert contract["sparse_encoding"]["sparsity"]["passes_sparsity"]
assert contract["decoder"] == [[3.0, 0.0], [0.0, 3.0]]
assert contract["comparison"]["beats_best_baseline"]
assert contract["stability"]["stable"]
assert contract["removal"]["random_removal_does_less"]
assert contract["audit"]["names_expected_cluster"]
contract

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "gelu-1l"
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603"
assert gpu["tokenizer_revision"] == "0f6671571a20be9756b9991d978047c03b75e749"
assert gpu["hook_name"] == "blocks.0.hook_resid_post"
assert gpu["activation_shape"] == [8, 512]
assert gpu["question_count"] == 4
assert gpu["pcd_row_count"] == 32
assert gpu["conditioned_concept_shape"] == [32, 32]
assert gpu["pcd_accuracy"] == 1.0
assert gpu["pcd_decoder_train_accuracy"] == 1.0
assert gpu["pcd_decoder_train_loss"] < 0.001
assert gpu["probe_accuracy"] == 0.5
assert gpu["best_baseline_accuracy"] == 0.5
assert gpu["question_shuffle_accuracy"] <= 0.25
assert gpu["pcd_seed_min_accuracy"] == 1.0
assert gpu["beats_best_baseline"]
assert gpu["passes_sparsity"]
assert gpu["stable"]
assert gpu["top_removal_changed"]
assert gpu["random_removal_does_less"]
assert gpu["random_removed_concept_active"]
assert gpu["names_expected_cluster"]
assert gpu["within_vram_budget"]
{key: gpu[key] for key in [
    "model_name",
    "question_count",
    "pcd_accuracy",
    "pcd_decoder_train_loss",
    "probe_accuracy",
    "best_baseline_accuracy",
    "question_shuffle_accuracy",
    "concept_density",
    "top_removal_delta",
    "random_removal_delta",
    "peak_vram_gb",
]}